In [ ]:
import socket
import threading
import pandas as pd
import signal
import sys

from scapy.all import sniff, IP, TCP
from datetime import datetime

captured_packets = []
stop_server_flag = False

def handle_client_connection(connection):

    try:

        connection.recv(1024)
        connection.close()

    except Exception:

        pass

def start_server_socket(ip_address="10.0.3.10", port=80):

    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

    try:

        server_socket.bind((ip_address, port))
        server_socket.listen(100)

        print(f"[AVVIO] Server socket in ascolto su {ip_address}:{port}...")

        while not stop_server_flag:

            server_socket.settimeout(1.0)

            try:

                connection, _ = server_socket.accept()

                thread = threading.Thread(

                    target=handle_client_connection,
                    args=(connection,)
                )

                thread.daemon = True
                thread.start()

            except socket.timeout:

                continue

    except Exception as error:

        print(f"[ERRORE] Server: {error}")

    finally:

        server_socket.close()

def process_packet_for_logging(packet):

    if packet.haslayer(IP) and packet.haslayer(TCP):

        if packet[TCP].dport == 80:

            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")

            captured_packets.append({

                "Timestamp": timestamp,
                "Sorgente": packet[IP].src,
                "Flags": str(packet[TCP].flags),
                "Taglia": len(packet)
            })

            source_ip = packet[IP].src
            destination_ip = packet[IP].dst
            source_port = packet[TCP].sport
            flags = str(packet[TCP].flags)
            packet_length = len(packet)

            try:

                payload = bytes(packet[TCP].payload)
                preview = payload[:100].decode(errors='replace')

                if len(payload) > 100:

                    preview += "..."

            except Exception:

                preview = "<payload non decodificabile>"

            print(f"[{timestamp}] {source_ip}:{source_port} -> {destination_ip}:80 | Flags: {flags} | Taglia: {packet_length}B | Payload: {preview}")

def save_and_exit(signal_number, frame):

    global stop_server_flag
    stop_server_flag = True

    print("\n[INFO] Arresto in corso, salvataggio dataset...")

    if captured_packets:

        data_frame = pd.DataFrame(captured_packets)
        data_frame.to_csv('/root/ricezioni_laptop.csv', index=False)

        print(f"[OK] Salvati {len(data_frame)} record su ricezioni_laptop.csv")

    sys.exit(0)

if __name__ == "__main__":

    signal.signal(signal.SIGINT, save_and_exit)

    server_thread = threading.Thread(target=start_server_socket)
    server_thread.daemon = True
    server_thread.start()

    print("Avvio ricevitore -> Stampo l'arrivo dei pacchetti in real time!")

    capture_filter = (
        
        "dst host 10.0.3.10 and tcp port 80 and "
        "(tcp[tcpflags] & tcp-syn != 0 or "
        "(tcp[tcpflags] & tcp-push != 0 and not "
        "(tcp[tcpflags] & tcp-ack != 0 and tcp[tcpflags] & tcp-push == 0)))"
    )

    sniff(iface="eth0", prn=process_packet_for_logging, filter=capture_filter, store=0)